# Sentiment analysis 

## 1. Textblob-FR

Documentation: https://textblob.readthedocs.io/en/dev/

### Imports

In [9]:
import os
import re
import random
import pandas as pd

txt_path = "../../data/txt"
year = 1964

files = [f for f in sorted(os.listdir(txt_path)) if str(year) in f]

phrases = []
for f in files:
    with open(os.path.join(txt_path, f), encoding="utf-8") as fh:
        texte = fh.read().replace("\n", " ")
    for s in re.split(r"(?<=[.!?])\s+", texte):
        if len(s) > 60:
            phrases.append(s.strip())

print(len(phrases), "phrases")

16013 phrases


In [26]:
# Tirage de 10 phrases au hasard
random.seed(2)
choisies = random.sample(phrases, 10)

for i, phrase in enumerate(choisies, start=1):
    print(f"{i}. {phrase}")

1. Edelen, qui portera bientôt sans doute les couleurs de l’Union Saint-Gilloise, ne s’estimait pas en forme suffisante pour prendre part au « Critérium des As », dont il termina deuxième l’an dernier.
2. Les dernières minutes virent les Anversois faire le forcing en vain car la solide défense locale ne se laissa pas déborder.
3. Il leur faudrait donner un sérieux coup de barre pour créer la surprise.
4. La toilette des bilans de fin d'année, autrement dit le * window circssing », amènerait également des rapatriements de capitaux.
5. Il sera également procédé à l’élection de deux membres pour figurer au sein du comité centrai.
6. — 20,30 h, 220, r, Belliard, « L'union des Eglises, une affaire do laïcs », par M.
7. donc sans retard dans sa progression scolaire, qu’il a renoncé ù poursuivre ses études.
8. 30, un bal de gala dans les locaux du K.M.A., avec le concours de l'orchestre du Casino de Knocke.
9. Pennoit, chef de cabinet du ministre; Huygens, secrétaire de cabinet; Delavaux, pré

In [1]:
from textblob import Blobber
from textblob_fr import PatternTagger, PatternAnalyzer

### Création d'une fonction `get_sentiment`

In [42]:
tb = Blobber(pos_tagger=PatternTagger(), analyzer=PatternAnalyzer())

def get_sentiment(input_text):
    blob = tb(input_text)
    polarity, subjectivity = blob.sentiment
    polarity_perc = f"{100*abs(polarity):.0f}"
    subjectivity_perc = f"{100*subjectivity:.0f}"
    if polarity > 0:
        polarity_str = f"{polarity_perc}% positive"
    elif polarity < 0:
        polarity_str = f"{polarity_perc}% negative"
    else:
        polarity_str = "neutral"
    if subjectivity > 0:
        subjectivity_str = f"{subjectivity_perc}% subjective"
    else:
        subjectivity_str = "perfectly objective"
    print(f"This text is {polarity_str} and {subjectivity_str}.")
    return polarity, subjectivity

### Analyser le sentiment 10 phrase

In [47]:
resultats = []
for i, phrase in enumerate(choisies, start=1):
    print(f"--- Phrase {i} ---")
    polarite, subjectivite = get_sentiment(phrase)
    resultats.append({
        "n": i,
        "phrase": phrase,
        "polarité": round(polarite, 2),
        "subjectivité": round(subjectivite, 2),
    })
resultats_df = pd.DataFrame(resultats)
resultats_df 

--- Phrase 1 ---
This text is 5% negative and 13% subjective.
--- Phrase 2 ---
This text is 2% negative and 32% subjective.
--- Phrase 3 ---
This text is 50% positive and 52% subjective.
--- Phrase 4 ---
This text is 10% positive and 23% subjective.
--- Phrase 5 ---
This text is 20% positive and 35% subjective.
--- Phrase 6 ---
This text is neutral and perfectly objective.
--- Phrase 7 ---
This text is 10% positive and perfectly objective.
--- Phrase 8 ---
This text is neutral and perfectly objective.
--- Phrase 9 ---
This text is neutral and perfectly objective.
--- Phrase 10 ---
This text is 15% positive and 10% subjective.


,n,phrase,polarité,subjectivité
0,1,"Edelen, qui portera bientôt sans doute les cou...",-0.05,0.13
1,2,Les dernières minutes virent les Anversois fai...,-0.02,0.32
2,3,Il leur faudrait donner un sérieux coup de bar...,0.50,0.53
3,4,"La toilette des bilans de fin d'année, autreme...",0.10,0.23
4,5,Il sera également procédé à l’élection de deux...,0.20,0.35
5,6,"— 20,30 h, 220, r, Belliard, « L'union des Egl...",0.00,0.00
6,7,"donc sans retard dans sa progression scolaire,...",0.10,0.00
7,8,"30, un bal de gala dans les locaux du K.M.A., ...",0.00,0.00
8,9,"Pennoit, chef de cabinet du ministre; Huygens,...",0.00,0.00
9,10,"SPORT 0 à 1000 ce : Moray (Anglia), 1 mn 48 s ...",0.15,0.10


## 2. Utilisation de transformers

Documentation: https://github.com/TheophileBlard/french-sentiment-analysis-with-bert

**!!** Si le code ne tourne pas sur votre machine, vous pouvez le tester directement sur Google Colab en utilisant [ce lien](https://colab.research.google.com/github/TheophileBlard/french-sentiment-analysis-with-bert/blob/master/colab/french_sentiment_analysis_with_bert.ipynb) **!!**

Le modèle peut également être testé en ligne sur [HuggingFace](https://huggingface.co/tblard/tf-allocine)

Note : `transformers` v5 a supprimé le support TensorFlow. On utilise donc ici `philschmid/pt-tblard-tf-allocine`, un portage PyTorch du même modèle, avec les classes `AutoModelForSequenceClassification` (au lieu de `TFAutoModelForSequenceClassification`).

### Installation des librairies et imports

In [5]:
!uv pip install sentencepiece
!uv pip install transformers>=5.19.0
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline

Using Python 3.13.9 environment at: /home/minasema/tac/.venv
Checked 1 package in 8ms
Using Python 3.13.9 environment at: /home/minasema/tac/.venv
Checked 1 package in 5ms


/home/minasema/tac/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Chargement du modèle

In [6]:
tokenizer = AutoTokenizer.from_pretrained("philschmid/pt-tblard-tf-allocine")
model = AutoModelForSequenceClassification.from_pretrained("philschmid/pt-tblard-tf-allocine")

sentiment_analyser = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 45054.25it/s]


### Analyser le sentiment d'une phrase

In [48]:
for i, phrase in enumerate(choisies, start=1):
    print(f"--- Phrase {i} ---")
    print(sentiment_analyser(phrase))

--- Phrase 1 ---
[{'label': 'NEGATIVE', 'score': 0.8946375846862793}]
--- Phrase 2 ---
[{'label': 'NEGATIVE', 'score': 0.8870941400527954}]
--- Phrase 3 ---
[{'label': 'NEGATIVE', 'score': 0.8428770899772644}]
--- Phrase 4 ---
[{'label': 'NEGATIVE', 'score': 0.5442095398902893}]
--- Phrase 5 ---
[{'label': 'NEGATIVE', 'score': 0.6082041263580322}]
--- Phrase 6 ---
[{'label': 'POSITIVE', 'score': 0.5252889394760132}]
--- Phrase 7 ---
[{'label': 'NEGATIVE', 'score': 0.7085297703742981}]
--- Phrase 8 ---
[{'label': 'POSITIVE', 'score': 0.5897961258888245}]
--- Phrase 9 ---
[{'label': 'NEGATIVE', 'score': 0.5020686984062195}]
--- Phrase 10 ---
[{'label': 'NEGATIVE', 'score': 0.726102352142334}]
